# QPM_BlackLitterman — Refactored

This notebook was refactored: core data / model / backtest logic moved to `src/` modules and `BacktestBL.py` (script).

Use this notebook as a lightweight demo that shows how to call the refactored functions. For the full walk‑forward monthly backtest, run `BacktestBL.py` from the repository root or run the cell below to execute it programmatically.

In [ ]:
# Install required packages (uncomment to run)
# !pip install pandas numpy yfinance pandas_datareader scikit-learn PyPortfolioOpt
print('Ready — this notebook uses the refactored modules in src/ and the script BacktestBL.py')

In [ ]:
# Minimal demo: import refactored helpers and compute a Black-Litterman allocation example
from src.data import get_prices, get_cleaned_data, get_market_caps
from src.bl import black_litterman_opt, markowitz_opt, compute_sigma

print('Imports OK — modules loaded from src/')

In [ ]:
# Example: small universe and a trivial view (identity P, small positive Q)
tickers = ['AAPL', 'MSFT', 'NVDA']
start_date = '2018-01-01'
prices = get_prices(tickers, start_date)
print('Downloaded prices — shape:', prices.shape)
market_caps = get_market_caps(tickers)
print('Market caps (sample):', market_caps)
# Build trivial view: small expected monthly returns (0.5% per month)
import numpy as np
P = np.eye(len(tickers))
Q = np.array([0.005] * len(tickers)).reshape(-1,1)
# Omega: small diagonal uncertainty
Omega = np.diag([1e-4] * len(tickers))

weights_bl = black_litterman_opt(tickers, prices, P, Q, Omega, market_caps, risk_free_rate=0.02)
print('Black-Litterman weights (clean):')
print(weights_bl)

# Markowitz baseline
weights_mk = markowitz_opt(tickers, prices, risk_free_rate=0.02)
print('
Markowitz (mean-historical) weights:')
print(weights_mk)

Notes and next steps:
- For the full monthly walk‑forward backtest (recommended), run `python BacktestBL.py` from the repository root.
- The script `BacktestBL.py` demonstrates how to produce P, Q, Omega from momentum or from multi‑linear regression and runs the walk‑forward loop.
- You can also run the walk‑forward programmatically from this notebook using the cell below — it is disabled by default to avoid accidental long runs.

In [ ]:
# Programmatic walk-forward runner (disabled by default)
# WARNING: running this cell will perform network downloads (yfinance/FRED) and may take several minutes.
# To run: set RUN_BACKTEST = True and execute the cell.
RUN_BACKTEST = False
if RUN_BACKTEST:
    # Import the walk_forward_backtest function from the BacktestBL script
    from BacktestBL import walk_forward_backtest
    # Example parameters — adjust as needed
    res = walk_forward_backtest(tickers=['AAPL', 'MSFT', 'NVDA', 'AMZN'], market='SPY', start_date='2018-01-01', method='multi_linear_regression')
    print('Portfolio metrics:')
    print(res.get('pf_metrics'))
else:
    print('Walk-forward backtest is disabled. Set RUN_BACKTEST = True to execute it (may take several minutes).')